<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/07-pandas/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Predicting Customer Churn from Raw Order Tables

A shop wants to know which customers are about to stop buying, so it can try to win them back before they're gone. Predicting this is called **churn prediction**, and it's one of the most common ML tasks in industry.

The interesting part isn't the model. It's turning raw business data, spread across several tables, into one row per customer with meaningful features, **without letting information from the future sneak in**. That's almost entirely pandas work, which makes it a perfect project for this module.

You'll:

1. generate a realistic set of related tables: customers, products, orders and order items
2. join them and explore the data
3. define the prediction problem precisely, with a **cutoff date**
4. build features per customer with `groupby`, date arithmetic and aligned one-hot encoding
5. train a logistic regression (in NumPy, from Module 6) and find out, honestly, whether it beats a simple business rule
6. see what happens when future data leaks into the features, and why the result is worthless
7. look at what the model learned, and how a shop would actually use it

## Step 1: The data

Real customer data is private, so we'll simulate a shop. The simulation is a small model of real behaviour: customers buy at different rates, premium members buy more, and before someone stops buying, their purchases usually slow down first. The model doesn't know any of these rules. It has to discover them from the tables, just as it would with real data.

The details of the simulation aren't the point of the project, so feel free to skim this cell.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2025)
N_CUSTOMERS = 2000
WINDOW_START, WINDOW_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-12-31")

# products
categories = ["Books", "Electronics", "Home", "Sports", "Beauty"]
base_price = {"Books": 15, "Electronics": 120, "Home": 40, "Sports": 35, "Beauty": 20}
products = pd.DataFrame({"product_id": [f"P{i:03d}" for i in range(40)],
                         "category": np.repeat(categories, 8)})
products["price"] = (products["category"].map(base_price) * rng.lognormal(0, 0.3, len(products))).round(2)

# customers
plan = rng.choice(["basic", "premium"], size=N_CUSTOMERS, p=[0.7, 0.3])
customers = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(N_CUSTOMERS)],
    "signup_date": pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 880, N_CUSTOMERS), unit="D"),
    "country": rng.choice(["Nepal", "India", "Japan", "Germany", "USA"], size=N_CUSTOMERS, p=[0.3, 0.25, 0.15, 0.15, 0.15]),
    "plan": plan,
})

# hidden behaviour: how often each customer buys, and when they stop
engagement = rng.lognormal(0, 0.6, N_CUSTOMERS) * np.where(plan == "premium", 1.6, 1.0)    # orders per month
active_from = customers["signup_date"].clip(lower=WINDOW_START)
monthly_hazard = np.clip(0.06 / engagement, 0.01, 0.5)
stop_date = active_from + pd.to_timedelta(rng.exponential(30 / monthly_hazard), unit="D")
favourite = rng.integers(0, len(categories), N_CUSTOMERS)

order_rows, item_rows, order_id = [], [], 0
for i in range(N_CUSTOMERS):
    first, last = active_from.iloc[i], min(stop_date.iloc[i], WINDOW_END)
    days = (last - first).days
    if days <= 0:
        continue
    n_candidates = rng.poisson(engagement[i] * days / 30)
    offsets = np.sort(rng.integers(0, days, n_candidates))
    when = first + pd.to_timedelta(offsets, unit="D")
    days_before_stop = (stop_date.iloc[i] - when).days
    keep = rng.random(n_candidates) > 0.8 * np.exp(-np.asarray(days_before_stop) / 45)   # slowing down before stopping
    for date in when[keep]:
        order_rows.append((order_id, customers["customer_id"].iloc[i], date))
        weights = np.full(len(categories), 1.0)
        weights[favourite[i]] = 4.0
        for _ in range(1 + rng.poisson(0.6)):
            cat = rng.choice(categories, p=weights / weights.sum())
            pid = rng.choice(products.loc[products["category"] == cat, "product_id"].to_numpy())
            item_rows.append((order_id, pid, int(rng.integers(1, 3))))
        order_id += 1

orders = pd.DataFrame(order_rows, columns=["order_id", "customer_id", "order_date"])
order_items = pd.DataFrame(item_rows, columns=["order_id", "product_id", "quantity"])

for name, table in [("customers", customers), ("products", products), ("orders", orders), ("order_items", order_items)]:
    print(f"{name:12s} {table.shape}")

customers    (2000, 4)
products     (40, 3)
orders       (36979, 3)
order_items  (58820, 3)


Four tables, linked by IDs, the way a real shop database is organised:

- **customers**: one row per customer
- **products**: one row per product, with its category and price
- **orders**: one row per order: who placed it, and when
- **order_items**: one row per product within an order, with the quantity

In [2]:
print(customers.head(3), "\n")
print(orders.head(3), "\n")
print(order_items.head(3), "\n")
print(products.head(3))

  customer_id signup_date country     plan
0       C0000  2025-01-07     USA    basic
1       C0001  2024-06-07     USA    basic
2       C0002  2024-02-29     USA  premium 

   order_id customer_id order_date
0         0       C0000 2025-01-07
1         1       C0000 2025-01-25
2         2       C0000 2025-02-10 

   order_id product_id  quantity
0         0       P014         2
1         0       P025         1
2         0       P037         2 

  product_id category  price
0       P000    Books   7.70
1       P001    Books  15.12
2       P002    Books  12.76


## Step 2: Join and explore

An order's value isn't stored anywhere directly. It has to be computed: join each item to its product to get the price, multiply by the quantity, and add up the items in each order. That's two merges and a `groupby`, with `validate` protecting each merge against duplicate keys:

In [3]:
items = order_items.merge(products, on="product_id", how="left", validate="many_to_one")
items["line_total"] = items["price"] * items["quantity"]

order_values = items.groupby("order_id").agg(
    amount=("line_total", "sum"),
    n_items=("quantity", "sum"),
    n_categories=("category", "nunique"),
)
orders_full = orders.merge(order_values, on="order_id", how="left", validate="one_to_one")

assert len(orders_full) == len(orders)
assert orders_full["amount"].notna().all()
orders_full.head()

,order_id,customer_id,order_date,amount,n_items,n_categories
0,0,C0000,2025-01-07,260.64,7,3
1,1,C0000,2025-01-25,24.16,1,1
2,2,C0000,2025-02-10,10.69,1,1
3,3,C0000,2025-03-03,65.40,2,1
4,4,C0000,2025-03-17,109.43,3,2


The two `assert`s check that the joins didn't add or lose any orders, and that every order got a value. Cheap insurance, as in exercise 10.

A first look at the business:

In [4]:
monthly = orders_full.groupby(orders_full["order_date"].dt.to_period("M")).agg(
    orders=("order_id", "size"),
    revenue=("amount", "sum"),
    customers=("customer_id", "nunique"),
)
print(monthly.round(0).to_string())

            orders   revenue  customers
order_date                             
2024-01       1147  124948.0        541
2024-02       1148  136385.0        548
2024-03       1284  141275.0        592
2024-04       1172  137117.0        580
2024-05       1363  157386.0        631
2024-06       1344  159595.0        599
2024-07       1515  175233.0        642
2024-08       1478  175665.0        677
2024-09       1538  184850.0        667
2024-10       1586  187549.0        705
2024-11       1500  162325.0        680
2024-12       1678  191552.0        729
2025-01       1637  184824.0        745
2025-02       1575  188332.0        730
2025-03       1779  203317.0        750
2025-04       1660  199453.0        770
2025-05       1911  214575.0        815
2025-06       1680  192255.0        752
2025-07       1855  217571.0        751
2025-08       1753  202183.0        717
2025-09       1684  207122.0        691
2025-10       1665  190676.0        684
2025-11       1555  183357.0        651


Grouping by `dt.to_period("M")` gives one row per calendar month. The number of active customers each month reflects both new signups and customers who've stopped.

In [5]:
category_revenue = items.groupby("category")["line_total"].sum().sort_values(ascending=False)
print((category_revenue / category_revenue.sum()).round(3))
print()
print(orders_full.merge(customers, on="customer_id").groupby("plan")["amount"].agg(["count", "mean"]).round(1))

category
Electronics    0.523
Home           0.189
Sports         0.154
Beauty         0.083
Books          0.051
Name: line_total, dtype: float64

         count   mean
plan                 
basic    21126  114.8
premium  15853  117.3


## Step 3: Define the problem precisely

"Predict churn" is too vague to build a model for. We need exact answers to three questions.

**When are we making the prediction?** We pick a **cutoff date**: 30 September 2025. We pretend today is that date. Features may only use information from before it.

**What counts as churn?** A customer churns if they place **no orders in the 90 days after the cutoff** (October to December 2025).

**Which customers do we predict for?** Only customers who were **active**: at least one order in the 180 days before the cutoff. Someone who stopped buying a year ago isn't "about to churn". They already have, and including them would make the task artificially easy.

In [6]:
CUTOFF = pd.Timestamp("2025-10-01")          # features use orders strictly before this date
LABEL_DAYS = 90
ACTIVE_DAYS = 180

past = orders_full[orders_full["order_date"] < CUTOFF]
future = orders_full[(orders_full["order_date"] >= CUTOFF) &
                     (orders_full["order_date"] < CUTOFF + pd.Timedelta(days=LABEL_DAYS))]

active_ids = past.loc[past["order_date"] >= CUTOFF - pd.Timedelta(days=ACTIVE_DAYS), "customer_id"].unique()
bought_later = set(future["customer_id"])

labels = pd.DataFrame({"customer_id": np.sort(active_ids)})
labels["churned"] = (~labels["customer_id"].isin(bought_later)).astype(int)

print(f"active customers at the cutoff: {len(labels)}")
print(f"churn rate: {labels['churned'].mean():.1%}")

active customers at the cutoff: 1107
churn rate: 25.3%


The `past` / `future` split is the most important line in the project. Everything about features comes from `past`. Only the label looks at `future`.

## Step 4: Build features

Now one row per active customer, describing their behaviour up to the cutoff. Each group of features is a hypothesis about what might signal churn:

- **RFM** (from exercise 11): how recently, how often and how much they bought
- **Trend**: are they buying less than they used to? We compare the last 90 days with the 90 days before that.
- **Basket**: average order value, items per order, how many categories they shop in
- **Customer profile**: how long they've been a customer, their plan, their country

In [7]:
def build_features(past: pd.DataFrame, customers: pd.DataFrame, ids, cutoff: pd.Timestamp) -> pd.DataFrame:
    p = past[past["customer_id"].isin(ids)]
    days_before = (cutoff - p["order_date"]).dt.days

    rfm = p.groupby("customer_id").agg(
        last_order=("order_date", "max"),
        frequency=("order_id", "size"),
        monetary=("amount", "sum"),
        avg_order_value=("amount", "mean"),
        avg_items=("n_items", "mean"),
        n_categories=("n_categories", "max"),
    )
    rfm["recency_days"] = (cutoff - rfm["last_order"]).dt.days

    recent = p[days_before < 90].groupby("customer_id").size()
    previous = p[(days_before >= 90) & (days_before < 180)].groupby("customer_id").size()
    rfm["orders_last_90d"] = recent.reindex(rfm.index, fill_value=0)
    rfm["orders_prev_90d"] = previous.reindex(rfm.index, fill_value=0)
    rfm["trend"] = (rfm["orders_last_90d"] - rfm["orders_prev_90d"]) / (rfm["orders_prev_90d"] + 1)

    profile = customers.set_index("customer_id").loc[rfm.index]
    rfm["tenure_days"] = (cutoff - profile["signup_date"]).dt.days
    rfm["premium"] = (profile["plan"] == "premium").astype(int)
    rfm["country"] = profile["country"]

    return rfm.drop(columns="last_order")


features = build_features(past, customers, labels["customer_id"], CUTOFF)
data = features.join(labels.set_index("customer_id"), how="inner", validate="one_to_one")
print(data.shape)
data.head()

(1107, 13)


,frequency,monetary,avg_order_value,avg_items,n_categories,recency_days,orders_last_90d,orders_prev_90d,trend,tenure_days,premium,country,churned
customer_id,,,,,,,,,,,,,
C0000,7,655.38,93.625714,3.142857,3,135,0,2,-0.666667,267,0,USA,1
C0001,49,8878.58,181.195510,2.346939,3,6,6,9,-0.300000,481,0,USA,0
C0004,21,2200.31,104.776667,2.333333,2,26,1,3,-0.500000,920,0,Nepal,0
C0006,23,1884.24,81.923478,2.217391,2,12,7,3,1.000000,499,0,Japan,0
C0007,10,1678.99,167.899000,1.900000,2,148,0,4,-0.800000,456,1,India,1


A few pandas details in there:

- `recent.reindex(rfm.index, fill_value=0)`: customers with no orders in a period don't appear in that `groupby` at all. Reindexing to the full list of customers puts them back with a 0, rather than leaving them as `NaN`. It's the same "decide the full set explicitly" idea as `monthly_pivot` and `one_hot_aligned`.
- `trend` divides by `orders_prev_90d + 1`, so customers with no earlier orders don't cause a division by zero.
- `customers.set_index("customer_id").loc[rfm.index]` looks up each customer's profile in the same order as the feature rows.
- `join(..., validate="one_to_one")` attaches the labels by index and checks that each customer appears exactly once on both sides.

Does anything already look different between customers who churned and those who didn't?

In [8]:
print(data.groupby("churned")[["recency_days", "frequency", "orders_last_90d", "trend", "tenure_days", "premium"]].mean().round(2).T)

churned               0       1
recency_days      21.59   93.32
frequency         28.60   14.24
orders_last_90d    5.79    1.13
trend              0.27   -0.30
tenure_days      495.58  495.94
premium            0.37    0.27


Churners last ordered longer ago, ordered less recently, and their trend is more negative: they were slowing down. Premium members churn less. These differences are what a model can learn from.

## Step 5: Split, encode, scale

We split customers into training and test sets, stratified by the label, with `groupby(...).sample` as in the lesson. Then one-hot encode `country` with the training columns as the definition (exercise 8), and standardize with statistics from the training set only.

In [9]:
test_df = data.groupby("churned").sample(frac=0.25, random_state=0)
train_df = data.drop(index=test_df.index)
print(f"train {len(train_df)}  test {len(test_df)}  churn rate train {train_df['churned'].mean():.1%}  test {test_df['churned'].mean():.1%}")

def to_matrix(train, test):
    tr = pd.get_dummies(train.drop(columns="churned"), columns=["country"], dtype=float)
    te = pd.get_dummies(test.drop(columns="churned"), columns=["country"], dtype=float)
    te = te.reindex(columns=tr.columns, fill_value=0.0)
    mean, std = tr.mean(), tr.std(ddof=0).replace(0, 1)
    return ((tr - mean) / std), ((te - mean) / std), list(tr.columns)

X_train_df, X_test_df, feature_names = to_matrix(train_df, test_df)
X_train, X_test = X_train_df.to_numpy(), X_test_df.to_numpy()
y_train, y_test = train_df["churned"].to_numpy(), test_df["churned"].to_numpy()
print(X_train.shape, X_test.shape)
print(feature_names)

train 830  test 277  churn rate train 25.3%  test 25.3%
(830, 16) (277, 16)
['frequency', 'monetary', 'avg_order_value', 'avg_items', 'n_categories', 'recency_days', 'orders_last_90d', 'orders_prev_90d', 'trend', 'tenure_days', 'premium', 'country_Germany', 'country_India', 'country_Japan', 'country_Nepal', 'country_USA']


`.to_numpy()` is the hand-off from pandas to NumPy: from here on, the model only sees plain arrays. That boundary is typical. pandas prepares the data, and the model (NumPy, scikit-learn or PyTorch) works on arrays.

## Step 6: Train and compare

The model is logistic regression, trained with gradient descent in NumPy. It's the binary version of the softmax regression from Module 6.

In [10]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))


def fit_logistic(X, y, lr=0.1, epochs=2000, l2=1e-3):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * (p - y).mean()
    return w, b


w, b = fit_logistic(X_train, y_train)
p_test = sigmoid(X_test @ w + b)
pred_test = (p_test >= 0.5).astype(int)

To judge it, we need baselines, and for churn, a natural one is a simple business rule: "anyone who hasn't ordered in the last 60 days is at risk".

In [11]:
def report(name, y_true, y_pred):
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    precision = tp / max(y_pred.sum(), 1)
    recall = tp / max(y_true.sum(), 1)
    print(f"{name:28s} accuracy {(y_pred == y_true).mean():6.1%}   precision {precision:6.1%}   recall {recall:6.1%}")


report("always 'stays'", y_test, np.zeros_like(y_test))
report("rule: no order in 60 days", y_test, (test_df["recency_days"].to_numpy() > 60).astype(int))
report("logistic regression", y_test, pred_test)

always 'stays'               accuracy  74.7%   precision   0.0%   recall   0.0%
rule: no order in 60 days    accuracy  87.4%   precision  79.7%   recall  67.1%
logistic regression          accuracy  87.4%   precision  84.3%   recall  61.4%


Precision and recall here are for the churn class: **precision** is how many of the customers we flag as churners really do churn, and **recall** is how many of the real churners we manage to flag.

The model and the simple rule end up very close. That's worth taking seriously rather than explaining away, because it's a common finding in churn prediction: **recency is an extremely strong signal**, and a single well-chosen threshold captures most of it. I tried several variations of the simulation and features while building this project, including a "days since last order compared with this customer's usual gap" feature and log-transformed inputs, and the result was the same each time: the model roughly ties the rule, sometimes slightly ahead, sometimes slightly behind.

So what does the model add? Three things, even when the headline numbers are similar:

- **The rule's 60-day threshold was picked with hindsight**, by someone who already knew recency matters. The model learns how much each signal matters from training data alone.
- **It gives a probability for every customer**, not a yes/no, which makes ranking possible (below).
- **It combines signals**, so if the shop's situation changes, say premium members start churning, retraining picks that up, whereas a fixed rule has to be noticed and rewritten by hand.

And the most important lesson of all: **always compare against a simple baseline**. Without the rule as a comparison, an 87% accurate model would look impressive, and you'd never know a one-line rule does just as well.

### How a shop would actually use this

A shop can't contact everyone. A more useful question is: *if we contact the 20% of customers the model thinks are most at risk, what share of the real churners do we reach?*

In [12]:
def capture_rate(y_true, scores, top_frac):
    n = int(round(len(scores) * top_frac))
    top = np.argsort(-scores)[:n]
    return y_true[top].sum() / y_true.sum()


for frac in [0.1, 0.2, 0.3]:
    model_cap = capture_rate(y_test, p_test, frac)
    rule_cap = capture_rate(y_test, test_df["recency_days"].to_numpy().astype(float), frac)
    print(f"contact top {frac:.0%}: model reaches {model_cap:5.1%} of churners, recency rule {rule_cap:5.1%}, random {frac:5.1%}")

contact top 10%: model reaches 37.1% of churners, recency rule 37.1%, random 10.0%
contact top 20%: model reaches 64.3% of churners, recency rule 62.9%, random 20.0%
contact top 30%: model reaches 80.0% of churners, recency rule 81.4%, random 30.0%


Contacting customers at random, you'd reach churners only in proportion to how many you contact. Ranking by the model's probabilities, or simply by recency, does about three times better than random for the top 20%. Again the two are close. This kind of table is how churn models are usually presented to the people who decide on budgets, and it's where you'd show whether a model is worth its extra complexity over the simple rule.

## Step 7: The leakage trap

Here's the mistake this project is really about. Suppose we'd built the features from **all** the orders, without the cutoff, because it seemed easier. Everything else stays exactly the same:

In [13]:
leaky_features = build_features(orders_full, customers, labels["customer_id"], CUTOFF)
leaky = leaky_features.join(labels.set_index("customer_id"), validate="one_to_one")

leaky_test = leaky.loc[test_df.index]
leaky_train = leaky.loc[train_df.index]
Xl_train_df, Xl_test_df, _ = to_matrix(leaky_train, leaky_test)
wl, bl = fit_logistic(Xl_train_df.to_numpy(), leaky_train["churned"].to_numpy())
leaky_pred = (sigmoid(Xl_test_df.to_numpy() @ wl + bl) >= 0.5).astype(int)

report("logistic regression (honest)", y_test, pred_test)
report("logistic regression (LEAKY)", y_test, leaky_pred)
print()
negative = leaky["recency_days"] < 0
print(f"customers with a negative recency: {negative[leaky['churned'] == 0].mean():.0%} of those who stayed, "
      f"{negative[leaky['churned'] == 1].mean():.0%} of churners")

logistic regression (honest) accuracy  87.4%   precision  84.3%   recall  61.4%
logistic regression (LEAKY)  accuracy  99.6%   precision 100.0%   recall  98.6%

customers with a negative recency: 100% of those who stayed, 0% of churners


The leaky model looks almost perfect. It's also completely useless.

With all orders included, every customer who stayed has a "last order" *after* the cutoff, since staying means ordering in the next 90 days, so their recency comes out negative. Churners almost never do. (The few that do are customers who came back only after the 90-day window.) So the sign of `recency_days` is practically the label itself, and the model has simply learned to read it. On 30 September, when you'd actually run the model, those future orders don't exist yet, so this signal isn't available.

Leakage is rarely this obvious in real projects. It hides in a feature computed over "all time", a customer attribute that's only filled in after they cancel, or a fill value computed over the whole dataset. The symptom is always the same: results that look too good. **If a model seems surprisingly accurate, suspect leakage before celebrating.** The defence is the discipline used here: one explicit cutoff, a `past` table for features, a `future` table only for labels, and a feature function that's only ever given `past`.

## Step 8: What did the model learn?

Because the features were standardized, the logistic regression weights are on comparable scales. Positive weights push towards "churn", negative ones towards "stays":

In [14]:
importance = pd.Series(w, index=feature_names).sort_values()
for name, weight in importance.items():
    bar = ("+" if weight > 0 else "-") * int(round(abs(weight) * 10))
    print(f"{name:18s} {weight:+.2f} {bar}")

orders_last_90d    -0.98 ----------
orders_prev_90d    -0.56 ------
trend              -0.25 ---
avg_order_value    -0.12 -
country_Germany    -0.11 -
n_categories       -0.11 -
country_Japan      -0.07 -
frequency          -0.04 
premium            -0.03 
monetary           -0.02 
country_Nepal      -0.01 
country_India      +0.00 
avg_items          +0.10 +
country_USA        +0.18 ++
tenure_days        +0.24 ++
recency_days       +1.02 ++++++++++


The picture matches what the simulation was built to do, which the model had no way of knowing:

- **Recency** is the strongest sign of churn: the longer since the last order, the higher the risk.
- **Recent activity** (`orders_last_90d`, and to a lesser extent `orders_prev_90d`) points strongly towards staying, and so does a positive **trend**. Customers whose buying is slowing down are the ones at risk.
- **Premium** has a small weight, even though premium members churn less: in step 4, they made up 37% of the customers who stayed but only 27% of the churners. That's because premium members churn less *because* they buy more often, and the activity features already capture that. Once those are in the model, the plan itself adds little.
- **Country** and **tenure** have no real effect in the simulation, so their weights are noise. Small weights like these on features that don't matter are normal, and a reminder not to read meaning into every number.

A caution about reading weights like this: correlated features share the credit between them. `frequency`, `monetary` and `orders_last_90d` all measure "how much do they buy", so the model can spread the weight across them in different ways. The overall story is reliable, but don't over-read the exact numbers for any single feature.

## Step 9: Save the features

Finally, save the feature table, so it can be reused without rebuilding it, with the cutoff recorded alongside:

In [15]:
import json
from pathlib import Path

out = Path("work") / "churn"
out.mkdir(parents=True, exist_ok=True)
data.reset_index().to_csv(out / "features_2025-10-01.csv", index=False)
(out / "features_2025-10-01.json").write_text(json.dumps({
    "cutoff": str(CUTOFF.date()),
    "label": f"no order in the {LABEL_DAYS} days after the cutoff",
    "population": f"at least one order in the {ACTIVE_DAYS} days before the cutoff",
    "rows": len(data),
    "churn_rate": round(float(data["churned"].mean()), 4),
    "features": [c for c in data.columns if c != "churned"],
}, indent=2), encoding="utf-8")

print(sorted(p.name for p in out.iterdir()))
print(pd.read_csv(out / "features_2025-10-01.csv").shape)

['features_2025-10-01.csv', 'features_2025-10-01.json']
(1107, 14)


Putting the cutoff date in the file name, and the full problem definition in a metadata file next to it, means nobody has to guess later which version of "churn" this table describes.

## Your turn

1. **A proper time-based test.** In real use, a model trained on one period is applied to a later one. Build features and labels at an earlier cutoff (say 1 July 2025), train on those, then test on the 1 October 2025 data. How does the score compare with the random split above? It's usually a little lower, and more honest.

2. **More features.** Add the number of distinct products bought, the share of spending in the customer's favourite category, or the average number of days between orders. Which ones improve the capture rate?

3. **Change the definition.** Try a 60-day or 120-day churn window. How do the churn rate and the model's performance change? Definitions are business decisions, and they change the whole problem.

4. **Tune the threshold.** Instead of 0.5, pick the threshold that gives 80% precision, because each retention offer costs money. How much recall do you give up?

5. **Find the leak yourself.** Add a feature that leaks more subtly, such as the customer's total number of orders over the whole period, and see how much it inflates the score. Then explain, in a sentence, why it's leakage.

6. **Use scikit-learn.** Train `sklearn.linear_model.LogisticRegression` on the same arrays and compare its predictions with yours.

## What you used

- **Several related tables** joined with `merge`, with `validate` and `assert`s to protect row counts
- **`groupby` with named aggregation** for order values, monthly trends and per-customer features
- **Date arithmetic** with `Timedelta`, `.dt.days` and `.dt.to_period`
- **`reindex(..., fill_value=0)`** to keep customers with no activity in a period
- **`isin`, boolean masks and `.loc`** to define the population and the label
- **Stratified sampling** with `groupby(...).sample`, and **aligned one-hot encoding**
- **Training-only scaling statistics**, then **`.to_numpy()`** to hand off to NumPy
- **Logistic regression** from scratch, compared with a majority baseline and a business rule

And the most important idea in applied ML: a precise problem definition with an explicit cutoff, and a strict separation between the past (for features) and the future (for labels).